# Black Marble Peru raster ESDA on an NVIDIA A100
Reproducible 2024-03-21 workflow using the tested `gpu_esda` wheel. The notebook contains orchestration only; stencil/statistical/permutation implementations remain in the package.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload dist/gpu_esda-0.2.0-py3-none-any.whl
wheel = next(name for name in uploaded if name.endswith('.whl'))
%pip install -q "{wheel}" "huggingface-hub>=0.25" "matplotlib>=3.8"


In [ ]:
import getpass, hashlib, json, os, platform, subprocess, time
from pathlib import Path
import cupy as cp, matplotlib.pyplot as plt, numpy as np, pandas as pd, pyarrow.parquet as pq
from google.colab import userdata
import gpu_esda
from gpu_esda import BlackMarbleRaster, RasterWeights, moran_global, moran_local, moran_observed
from gpu_esda.backend import to_numpy
from gpu_esda.raster.io import write_json, write_local_parquet
from gpu_esda.visualization import save_figure
props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props['name'].decode()
assert 'A100' in gpu_name, f'An NVIDIA A100 is required; detected {gpu_name}'
free, total = cp.cuda.Device().mem_info
try: HF_TOKEN = userdata.get('HF_TOKEN')
except Exception: HF_TOKEN = getpass.getpass('Hugging Face read token: ')
assert HF_TOKEN, 'A private-repository read token is required'
OUT = Path('/content/results'); FIG = OUT/'figures'; BM = OUT/'blackmarble'
FIG.mkdir(parents=True, exist_ok=True); BM.mkdir(parents=True, exist_ok=True)
environment = {'gpu': gpu_name, 'vram_bytes': total, 'driver': cp.cuda.runtime.driverGetVersion(), 'cuda_runtime': cp.cuda.runtime.runtimeGetVersion(), 'python': platform.python_version(), 'cupy': cp.__version__, 'gpu_esda': gpu_esda.__version__, 'package_commit': 'd7f15b1'}
environment


In [ ]:
from huggingface_hub import hf_hub_download
REPO='faviolc/ESDAGPU'; EXPECTED=6_111_958
GRID_NAME='data/grid/grid_peru_vnp46a2.parquet'
DAILY_NAME='data/daily/year=2024/month=03/blackmarble_peru_2024-03-21.parquet'
grid_path=Path(hf_hub_download(REPO, GRID_NAME, repo_type='dataset', token=HF_TOKEN))
daily_path=Path(hf_hub_download(REPO, DAILY_NAME, repo_type='dataset', token=HF_TOKEN))
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
assert pq.ParquetFile(grid_path).metadata.num_rows == EXPECTED
assert pq.ParquetFile(daily_path).metadata.num_rows == EXPECTED
assert sha256(grid_path) == '40744b0152a4ef82085250107ca7d665ccb40c06746e9476c156fbcc9de2ce93'
assert sha256(daily_path) == 'b39dbb8505f609a49c4013dc7911bbc29aea54f90487bd07f5555ec86e354142'
required_grid={'cell_id','grid_row','grid_col','lon','lat'}; required_daily={'cell_id','ntl','mandatory_quality_flag','latest_high_quality_retrieval'}
assert required_grid <= set(pq.read_schema(grid_path).names)
assert required_daily <= set(pq.read_schema(daily_path).names)
{'grid_bytes':grid_path.stat().st_size,'daily_bytes':daily_path.stat().st_size}


In [ ]:
run_started=time.perf_counter(); raster=BlackMarbleRaster.from_parquet(grid_path,daily_path,value_column='ntl',dtype='float32')
assert raster.metadata['source_rows']==EXPECTED and raster.values.shape==(4395,3042)
assert raster.metadata['missing_values']==0 and np.count_nonzero(raster.values[raster.mask]==0)>0
# Find a populated 128x128 validation window; calculations still use its exact mask.
window=None
for r in range(0,raster.values.shape[0]-128,128):
    for c in range(0,raster.values.shape[1]-128,128):
        m=raster.mask[r:r+128,c:c+128]
        if m.sum()>2000: window=(r,c); break
    if window: break
r,c=window; small=raster.values[r:r+128,c:c+128]; small_mask=raster.mask[r:r+128,c:c+128]
cpu_w=RasterWeights.queen(small_mask,backend='cpu',dtype='float64'); gpu_w=RasterWeights.queen(small_mask,backend='gpu',dtype='float64')
cpu_g,cpu_l=moran_observed(small.astype('float64'),cpu_w); gpu_g,gpu_l=moran_observed(small.astype('float64'),gpu_w)
assert abs(cpu_g.I-gpu_g.I)<1e-10
np.testing.assert_allclose(cpu_l.local_i,to_numpy(gpu_l.local_i),atol=1e-10,rtol=1e-8,equal_nan=True)
validation={'window':[r,c,128,128],'cpu_I':cpu_g.I,'gpu_I':gpu_g.I}
validation


In [ ]:
def raster_figure(values,title,name,cmap='magma'):
    fig,ax=plt.subplots(figsize=(8,10)); im=ax.imshow(np.ma.masked_invalid(values),cmap=cmap,interpolation='nearest'); ax.set(title=title); fig.colorbar(im,ax=ax,shrink=.7); save_figure(fig,name,FIG); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,10)); ax.imshow(raster.structural_mask,cmap='gray',interpolation='nearest'); ax.set(title='Black Marble Peru grid coverage'); save_figure(fig,'grid_coverage.png',FIG); plt.close(fig)
raster_figure(raster.values,'NTL raw — 2024-03-21','ntl_raw_map.png')
log_host=np.full(raster.values.shape,np.nan,np.float32); log_host[raster.mask]=np.log1p(raster.values[raster.mask])
raster_figure(log_host,'log1p(NTL) — 2024-03-21','ntl_log1p_map.png')


In [ ]:
transfer_start=time.perf_counter(); raw_gpu=cp.asarray(raster.values,dtype=cp.float32); mask_gpu=cp.asarray(raster.mask); cp.cuda.Stream.null.synchronize(); transfer_seconds=time.perf_counter()-transfer_start
log_gpu=cp.log1p(raw_gpu); cp.cuda.Stream.null.synchronize()
observed_records=[]; queen_operator=None; queen_log_local=None
for variable,values in [('ntl',raw_gpu),('log1p',log_gpu)]:
    for stencil in ['rook','queen','d2_r2']:
        op=(RasterWeights.rook(mask_gpu,backend='gpu') if stencil=='rook' else RasterWeights.queen(mask_gpu,backend='gpu') if stencil=='queen' else RasterWeights.inverse_distance(mask_gpu,radius=2,backend='gpu'))
        g,l=moran_observed(values,op)
        counts=cp.bincount(l.quadrant_code[l.mask].astype(cp.int32),minlength=5).get()
        observed_records.append({'variable':variable,'stencil':stencil,'moran_i':g.I,'n':g.n,'islands':int(l.island.sum().item()),'HH':int(counts[1]),'LH':int(counts[2]),'LL':int(counts[3]),'HL':int(counts[4]),**{f't_{k}':v for k,v in g.timings.items()}})
        if variable=='log1p' and stencil=='queen': queen_operator=op; queen_log_local=l
observed_records


In [ ]:
# Safe smoke inference first. Local inference is conditional and chunked; no n×R array is created.
global_99=moran_global(log_gpu,queen_operator,permutations=99,seed=12345)
local_99=moran_local(log_gpu,queen_operator,permutations=99,seed=12345,fdr=True)
smoke={'global':global_99.to_dict(),'local_significant_fdr':int(local_99.significant.sum().item()),'free_vram_after':int(cp.cuda.Device().mem_info[0])}
RUN_999=True
safe_for_999=cp.cuda.Device().mem_info[0] > 12*2**30
if RUN_999 and safe_for_999:
    del local_99; cp.get_default_memory_pool().free_all_blocks()
    global_final=moran_global(log_gpu,queen_operator,permutations=999,seed=12345)
    local_final=moran_local(log_gpu,queen_operator,permutations=999,seed=12345,fdr=True)
else:
    global_final,local_final=global_99,local_99
smoke,global_final.p_sim,global_final.permutations


In [ ]:
global_payload={'date':'2024-03-21','product':'VNP46A2','observed':observed_records,'inference_queen_log1p':global_final.to_dict()}
write_json(global_payload,BM/'blackmarble_2024-03-21_global.json')
write_local_parquet(local_final,raster,BM/'blackmarble_2024-03-21_local_queen.parquet')
benchmark=pd.DataFrame(observed_records); benchmark['transfer_seconds']=transfer_seconds; benchmark.to_csv(BM/'blackmarble_2024-03-21_a100_benchmarks.csv',index=False)
metadata={**environment,'validation':validation,'grid_sha256':sha256(grid_path),'daily_sha256':sha256(daily_path),'rows':EXPECTED,'shape':list(raster.values.shape),'total_seconds':time.perf_counter()-run_started,'smoke_99':smoke}
write_json(metadata,BM/'blackmarble_2024-03-21_run_metadata.json')
local_i_map=to_numpy(local_final.local_i).astype(float); local_i_map[~raster.mask]=np.nan
codes=to_numpy(local_final.quadrant_code).astype(float); codes[~raster.mask]=np.nan
raster_figure(local_i_map,'Local Moran Queen — log1p(NTL)','local_moran_queen.png','coolwarm'); raster_figure(codes,'LISA clusters Queen — log1p(NTL)','lisa_clusters_queen.png','tab10')
fig,ax=plt.subplots(figsize=(8,4)); benchmark.set_index(['variable','stencil'])['t_spatial_lag'].plot.bar(ax=ax); ax.set_ylabel('seconds'); ax.set_title('A100 raster lag runtime'); save_figure(fig,'cpu_gpu_runtime_comparison.png',FIG); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4)); ax.bar(['used','free'],[(total-cp.cuda.Device().mem_info[0])/2**30,cp.cuda.Device().mem_info[0]/2**30]); ax.set_ylabel('GiB'); ax.set_title('A100 memory after workflow'); save_figure(fig,'gpu_memory_profile.png',FIG); plt.close(fig)
import shutil
archive=shutil.make_archive('/content/blackmarble_peru_2024-03-21_results','zip',OUT)
{'archive':archive,'files':[str(p) for p in sorted(OUT.rglob('*')) if p.is_file()]}
